#**0. INTRODUCTION**

In this project, we try to understand one of IEA’s most important datasets, the Monthly Electricity Statistics (MES) provided in the file MES_0725.csv. The file contains monthly data (in GWh) from January 2010 – July 2025 that show electricity production, imports, exports, final consumption, losses, energy sources (such as coal, natural gas, hydro, solar, and wind) data from each country. This will allow us to explore how energy is supplied and used over time, using accurate, comparable figures from different countries.

Then, we have implemented together is fully aligned with the materials provided throughout the bootcamp sessions S1 to S16, covering fundamental topics of python such as manipulating datatypes, printing values, working with variables, arrays, defining functions, conditioning using if, else, elif, Loops using for, Working with lists, sets, Using the import functionality to read files, manipulation libraries like datetime, numpy, pandas, and visualization libraries like matplotlib, plotly and ipywidgets to make the visualizations interactive.

#**1. DATA SETUP & CONFIGURATION**

This stage includes importing data source from Google Drive and import of Python Libraries.

In [ ]:
#Libraries involved in core data manipulation:
import pandas as pd
import numpy as np

#Libraries used for display utilities:
from IPython.display import display, clear_output

#Libraries used for visualization:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

#For data preprocessing:
from sklearn.preprocessing import MinMaxScaler

# Libraries for interactive widgets:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, Dropdown

#Import data:
url = 'https://drive.google.com/uc?id=14-nTQ8VBM12z8eZ1Y47nFM_Q8W20FLA5'
#Read data to dataframe using pandas library:
df = pd.read_csv(url)

#Check the top 5 records in the df:
print(df.head())

     Country    Time                     Balance  \
0  Australia  Jul-25  Net Electricity Production   
1  Australia  Jul-25  Net Electricity Production   
2  Australia  Jul-25  Net Electricity Production   
3  Australia  Jul-25  Net Electricity Production   
4  Australia  Jul-25  Net Electricity Production   

                             Product       Value Unit  
0                        Electricity  24847.3008  GWh  
1                              Hydro   1108.3607  GWh  
2            Total Combustible Fuels  16131.3856  GWh  
3  Coal, Peat and Manufactured Gases  11233.6792  GWh  
4         Oil and Petroleum Products    419.2780  GWh  


#**2. DATA CLEANING AND CLASSIFICATION**

### **2. a. Remove Duplicate Values**
- Here we check for duplicate datapoints from our dataset.

In [ ]:
print(df.duplicated().sum())

0


### **2. b. Handle Null Values**
- Next, we check and remove all the null values from the dataset


In [ ]:
df = df.dropna()

### **2. c. Standardize the Data Formatting**
- Standardizing the date and time columns from our dataset

In [ ]:
df['Time'] = pd.to_datetime(df['Time'], format='%b-%y', errors='coerce')

### **2. d. Remove Excess datapoints**

#### **i. Remove Sub-Total and Grouped values from Country**

To ensure that only individual country-level data remains, we removed all rows representing subtotal or grouped values such as OECD totals or regional aggregates.

In [ ]:
#Declare an array of keywords to be dropped from the df:
keywords = ['OECD', 'Total']

#Delete records using for loop:
for word in keywords:
    df = df[~df['Country'].str.contains(word, case=False, na=False)]



#### **ii. Remove Aggregated Totals from Product**
- To focus on individual energy products, we removed rows representing aggregated totals such as Total Combustible Fuels and Total Renewables (Hydro, Geo, Solar, Wind, Other) from the dataset.


In [ ]:
#Declare an array of keywords to be dropped from the df:
keywords = ['Total Combustible Fuels', 'Total Renewables (Hydro, Geo, Solar, Wind, Other)']

#Delete records using for loop:
for word in keywords:
    df = df[~df['Product'].str.contains(word, case=False, na=False, regex=False)]

#### **iii. Drop records from the year 2025**
- Finally, we removed all datapoints corresponding to the year 2025.

In [ ]:
df = df[df['Time'].dt.year < 2025].reset_index(drop=True)

### **2. e. Data Type synchronization**
- Next, we set the datatype as string for Country, Balance, Product and Unit columns.


In [ ]:
df['Country'] = df['Country'].astype("string")
df['Balance'] = df['Balance'].astype("string")
df['Product'] = df['Product'].astype("string")
df['Unit'] = df['Unit'].astype("string")

### **2. f. Data Checking for Negative Value**
- We dropped datapoints with negative values if any.

In [ ]:
df = df.dropna(subset=['Value'])

### **2. g. Classification of Countries AS Core and Partner Countries**

- We create a new data column "Group" to segregate the countries as Core or Partner Members.


In [ ]:
#Define core member countries of OPEC using this array:
core_members = [
    "Australia", "Austria", "Belgium", "Canada", "Chile", "Colombia", "Costa Rica",
    "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Ireland", "Italy", "Japan", "Korea", "Latvia", "Lithuania",
    "Luxembourg", "Mexico", "Netherlands", "New Zealand", "Norway", "Poland", "Portugal",
    "Republic of Turkiye","Slovak Republic", "Slovenia", "Spain", "Sweden", "Switzerland",
    "United Kingdom", "United States"
]

#Label all countries in the df as Core or Key Partner:
df['Group'] = df['Country'].apply(lambda x: 'Core Member' if x in core_members else 'Key Partner')

#Set datatype of this new column as String:
df['Group'] = df['Group'].astype("string")

### **2. i. Classify Energy Type as Renewable and Non-Renewable sources**

- We created new column "Energy_Type" to classify renewable and non-renewable sources.


In [ ]:
#Renewable sources array:
renewables = [
    "Hydro", "Geothermal", "Wind", "Solar", "Other Renewables", "Combustible Renewables"
]

#Non-renewable sources array:
non_renewables = [
    "Coal, Peat and Manufactured Gases", "Natural Gas", "Oil and Petroleum Products",
    "Other Combustible Non-Renewables", "Nuclear"
]

#Others:
neutral_or_unspecified = ["Electricity", "Not Specified"]

#Function used to categorise the records as Renewable, Non-Renewable, Others:
def classify_energy_type(product):
    if product in renewables:
        return "Renewable"
    elif product in non_renewables:
        return "Non-Renewable"
    elif product in neutral_or_unspecified:
        return "Unspecified"
    else:
        return "Unknown"

df['Energy Type'] = df['Product'].apply(classify_energy_type)

#Set datatype as string:
df['Energy Type'] = df['Energy Type'].astype("string")

### **2. j. Summary of Data**
- Generate and display descriptive statistics for each Balance category in the dataset

In [ ]:
stats_by_balance = df.groupby('Balance')['Value'].describe().T.round(2)
print(stats_by_balance)

Balance  Distribution Losses  Final Consumption (Calculated)  \
count                6480.00                         6480.00   
mean                 1514.25                        22759.00   
std                  3482.31                        54866.51   
min                     0.00                          435.96   
25%                   190.46                         2888.18   
50%                   397.78                         6395.39   
75%                  1792.92                        21710.84   
max                 33401.80                       412844.24   

Balance  Net Electricity Production  Total Exports  Total Imports  \
count                      85271.00        5791.00        5786.00   
mean                        6043.70        1252.12        1270.10   
std                        34632.44        1635.23        1263.22   
min                            0.00           0.00           0.00   
25%                           13.94         186.02         440.32   
50%      

#**3. DATA ANALYSIS AND VISUALIZATION**

- After completing all data cleaning and validation, we created a staging dataset to store the processed data. This allows the cleaned data to be easily reused for further analysis or other operations.

### **3. a. Net Energy Balance - Analysis**

- Conducted analysis to identify energy surpluses or deficits for each country. This incorporates data on production, imports, consumption, and exports
- Filter for relevant Balance types and Core Member Countries
- Calculate and define the Quarter period
- Group by Country and Quarter to calculate the sum of value for each Balance type
- Caulcate Total Generation and Consumption
- Calculate Net Energy Balance standalone value and then Net Energy Balance as a percentage of Total Generation


In [ ]:
#Intermediate df created by copying required columns from the main df:
net_energy_df = df[df['Balance'].isin([
    'Net Electricity Production',
    'Total Imports',
    'Final Consumption (Calculated)',
    'Total Exports'
])].copy()

#Filter only Core Member Countries:
net_energy_df = net_energy_df[net_energy_df['Country'].isin(core_members)].copy()

#Declare Quarter by reading the value of Time column in the dataframe:
net_energy_df['Quarter'] = net_energy_df['Time'].dt.to_period('Q')

#Group by the Value over Country, Quarter, and Balance. Create a new df called grouped:
grouped = net_energy_df.groupby(['Country', 'Quarter', 'Balance'])['Value'].sum().unstack()

#Total Generation = Net Electricity Production + Total Imports from the grouped df:
grouped['Total Generation'] = grouped['Net Electricity Production'].fillna(0) + grouped['Total Imports'].fillna(0)

#Total Consumption = Final Consumption (Calculated) - Total Exports from the grouped df:
grouped['Total Consumption'] = grouped['Final Consumption (Calculated)'].fillna(0) + grouped['Total Exports'].fillna(0)

#Net Energy Balance = Total Generation - Total Consumption:
grouped['Net Energy Balance'] = grouped['Total Generation'] - grouped['Total Consumption']

#Net Energy Balance % = Net Energy Balance / Total Generation * 100:
grouped['Net Energy Balance (%)'] = (grouped['Net Energy Balance'] / grouped['Total Generation'].replace(0, np.nan)) * 100

#Round off the final value of Net Energy Balance % to 2 decimal places:
grouped['Net Energy Balance (%)'] = grouped['Net Energy Balance (%)'].round(2)

#Append the Calculated Net Energy Balance and Net Energy Balance % values to the first df:
df_net_energy_balance = grouped[['Total Generation', 'Total Consumption', 'Net Energy Balance', 'Net Energy Balance (%)']].reset_index()

display(df_net_energy_balance.head())

Balance,Country,Quarter,Total Generation,Total Consumption,Net Energy Balance,Net Energy Balance (%)
0,Australia,2010Q1,118701.144,55230.487,63470.657,53.47
1,Australia,2010Q2,116095.627,53984.108,62111.519,53.50
2,Australia,2010Q3,124053.440,57687.991,66365.449,53.50
3,Australia,2010Q4,114399.794,53188.443,61211.351,53.51
4,Australia,2011Q1,127338.903,59537.443,67801.460,53.24


### **3. b. Data Visualization of Net Energy Balance**
- Extract year from the Quarter
- Add a dropdown widget
- Visualization code


In [ ]:
#Extract Year from the Quarters column of the Net Energy Balance df:
df_net_energy_balance['Year'] = df_net_energy_balance['Quarter'].apply(lambda x: x.year)

#Create a list of all the Years and 'Average' for Dropdown values:
years = ['Average'] + sorted(df_net_energy_balance['Year'].unique())

#Create Year dropdown, default value is Average:
year_dropdown = Dropdown(options=years, description='Select Year:', value='Average')

#Create a function to update the plot when the user changes the dropdown value:
def update_plot(year):
    clear_output(wait=True)

    #Check the selected value of the Year from the dropdown:
    if year == 'Average':
        filtered_df = df_net_energy_balance.copy()
        title_year = 'All Years (Average)'
    else:
        filtered_df = df_net_energy_balance[df_net_energy_balance['Year'] == year].copy()
        title_year = f'Year {year}'

    #Take the average of Net Energy Balance values:
    mean_net_energy_balance = filtered_df.groupby('Country')[['Net Energy Balance', 'Net Energy Balance (%)']].mean().reset_index()
    #Sort the df in descending:
    mean_net_energy_balance = mean_net_energy_balance.sort_values('Net Energy Balance', ascending=False)

    #Create blank figure:
    fig, ax1 = plt.subplots(figsize=(15, 8))

    #Plot a bar chart showing the average Net Energy Balance (in GWh) for each country:
    ax1.bar(mean_net_energy_balance['Country'], mean_net_energy_balance['Net Energy Balance'], color='skyblue')

    #Set the X-axis label as 'Country' and Y-axis label as 'Average Net Energy Balance (GWh)':
    ax1.set_xlabel('Country', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Average Net Energy Balance (GWh)', fontsize=12, fontweight='bold')

    #Customize label colors, font size, and weight for better readability and visualization:
    ax1.tick_params(axis='y')

    #Set x-axis ticks to match the number of countries and label them with country names:
    ax1.set_xticks(np.arange(len(mean_net_energy_balance['Country'])))
    #Rotate x-axis labels for better readability:
    ax1.set_xticklabels(mean_net_energy_balance['Country'], rotation=90)
    #Adjust y-axis limits to start at 0 and extend 20% beyond the maximum Net Energy Balance value for clear visualization:
    ax1.set_ylim(0, mean_net_energy_balance['Net Energy Balance'].max() * 1.2)

    ax1.grid(axis='y', alpha=0.3, linestyle='--')

    #Create a secondary Y-axis (ax2) sharing the same X-axis to plot Net Energy Balance (%) for each country:
    ax2 = ax1.twinx()
    #Plot percentage values as a red line with circle markers to show relative balance trends:
    ax2.plot(mean_net_energy_balance['Country'], mean_net_energy_balance['Net Energy Balance (%)'],
             color='red', marker='o', linestyle='-', linewidth=2, markersize=6)
    ax2.set_ylabel('Average Net Energy Balance (%)', color='red', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='red')

    plt.title(f'Average Net Energy Balance and Percentage by Country (Core Members) - {title_year}',
              fontsize=14, fontweight='bold', pad=20)
    #Display the final interactive plot allowing users to change the year dynamically using the dropdown:
    plt.tight_layout()
    plt.show()

interact(update_plot, year=year_dropdown);

interactive(children=(Dropdown(description='Select Year:', options=('Average', np.int64(2010), np.int64(2011),…

### **3. c. Import Dependecy - Analysis**

- The import dependency analysis used to calculate data on imports and consumption.
- Filter data to extract only Total Import and Final Consumption related values from the original dataframe

In [ ]:
#Create an intermediate df copying the required data from the original datafram:
import_dependency_df = df[
    df['Country'].isin(core_members) &
    df['Balance'].isin(['Total Imports', 'Final Consumption (Calculated)'])
].copy()

#Declare Quarter from Time in our dataframe:
import_dependency_df['Quarter'] = import_dependency_df['Time'].dt.to_period('Q')

#Create a pivot to a create a summary table:
import_dependency_pivot = import_dependency_df.pivot_table(
    index=['Country', 'Quarter'],
    columns='Balance',
    values='Value',
    aggfunc='sum'
).reset_index()

#Calculate import dependency as a percentage:
import_dependency_pivot['Import Dependency'] = (
    import_dependency_pivot['Total Imports'].fillna(0) /
    import_dependency_pivot['Final Consumption (Calculated)'].fillna(0) * 100
)

#Copy relevant data in a new df:
df_import_dependency = import_dependency_pivot[['Country', 'Quarter', 'Import Dependency']].copy()

#Round the dependency value to 2 decimal places:
df_import_dependency['Import Dependency'] = df_import_dependency['Import Dependency'].round(2)

#Display the results df:
display(df_import_dependency.head())

Balance,Country,Quarter,Import Dependency
0,Australia,2010Q1,0.0
1,Australia,2010Q2,0.0
2,Australia,2010Q3,0.0
3,Australia,2010Q4,0.0
4,Australia,2011Q1,0.0


### **3. d. Sources of Energy - Analysis**

- Analysis for different sources of energies
- Filter the DataFrame for 'Net Electricity Production' balance and Core Members
- Calculate Quarter from dates
- Group the Sum of Vaules by Country Quarter and Energy Type

In [ ]:
#Create an intermediate df:
generation_df = df[
   (df['Country'].isin(core_members)) &
   (df['Balance'] == 'Net Electricity Production')
].copy()

#Derive Quarters from existing Time column:
generation_df['Quarter'] = generation_df['Time'].dt.to_period('Q')

#Group the generation data and unstack the 'Energy Type' column to create separate columns for Renewable and Non-Renewable:
grouped_generation = generation_df.groupby(['Country', 'Quarter', 'Energy Type'])['Value'].sum().unstack()

#Rename columns for better clarity:
grouped_generation = grouped_generation.rename(columns={
    'Renewable': 'Total Renewable Generation',
    'Non-Renewable': 'Total Non-renewable Generation'
}).fillna(0)

#Reset the index for a clean structure:
df_generation_summary = grouped_generation[['Total Renewable Generation', 'Total Non-renewable Generation']].reset_index()

#Display the result df:
display(df_generation_summary.head())

Energy Type,Country,Quarter,Total Renewable Generation,Total Non-renewable Generation
0,Australia,2010Q1,4816.806,54533.765
1,Australia,2010Q2,5443.164,52604.648
2,Australia,2010Q3,6965.534,55061.185
3,Australia,2010Q4,7385.356,49814.540
4,Australia,2011Q1,5914.957,57754.494


### **3. e. Sources of Energy - Visualization**

- Aggregate quarterly data to annual data
- Map country codes with the exact country names mentioned in our dataset
- Set the trace for Average to be selected by Default in the visualization
- Create a trace for all the other years
- Configure dropdown values


In [ ]:
#Create an intermediate df with Quarter to analyse QoQ data:
df_generation_summary['Year'] = df_generation_summary['Quarter'].astype(str).str[:4].astype(int)

#Copy Country, Year and the sum of Total Renewable Generation into df_annual dataframe:
df_annual = df_generation_summary.groupby(['Country', 'Year']).agg({
    'Total Renewable Generation': 'sum'
}).reset_index()

#Create a separate df with average values for the reference axes of the quadrants:
df_average = df_generation_summary.groupby(['Country']).agg({
    'Total Renewable Generation': 'sum'
}).reset_index()

#Sort the Years and ensure that only distinct values of years are present:
years = sorted(df_annual['Year'].unique())

#Map the Country names in our df with their equivalent codes:
country_code_mapping = {
    "Australia": "AUS", "Austria": "AUT", "Belgium": "BEL", "Canada": "CAN",
    "Chile": "CHL", "Colombia": "COL", "Costa Rica": "CRI", "Czech Republic": "CZE",
    "Denmark": "DNK", "Estonia": "EST", "Finland": "FIN", "France": "FRA",
    "Germany": "DEU", "Greece": "GRC", "Hungary": "HUN", "Iceland": "ISL",
    "Ireland": "IRL", "Italy": "ITA", "Japan": "JPN", "Korea": "KOR",
    "Latvia": "LVA", "Lithuania": "LTU", "Luxembourg": "LUX", "Mexico": "MEX",
    "Netherlands": "NLD", "New Zealand": "NZL", "Norway": "NOR", "Poland": "POL",
    "Portugal": "PRT", "Republic of Turkiye": "TUR", "Slovak Republic": "SVK",
    "Slovenia": "SVN", "Spain": "ESP", "Sweden": "SWE", "Switzerland": "CHE",
    "United Kingdom": "GBR", "United States": "USA"
}

df_annual['Country_Code'] = df_annual['Country'].map(country_code_mapping)
df_average['Country_Code'] = df_average['Country'].map(country_code_mapping)

#
fig = go.Figure()

trace_avg = go.Choropleth(
    locations=df_average['Country_Code'],
    z=df_average['Total Renewable Generation'],
    text=df_average['Country'],
    colorscale='Greens',
    autocolorscale=False,
    reversescale=False,
    marker_line_color='darkgray',
    marker_line_width=0.5,
    colorbar=dict(
        title="Total Renewable<br>Generation (GWh)",
        tickformat=","
    ),
    hovertemplate='<b>%{text}</b><br>' +
                  'Renewable Generation: %{z:,.0f} GWh<br>' +
                  '<extra></extra>',
    visible=True
)
fig.add_trace(trace_avg)

#Loop over the years
for year in years:
    year_data = df_annual[df_annual['Year'] == year].copy()

    trace = go.Choropleth(
        locations=year_data['Country_Code'],
        z=year_data['Total Renewable Generation'],
        text=year_data['Country'],
        colorscale='Greens',
        autocolorscale=False,
        reversescale=False,
        marker_line_color='darkgray',
        marker_line_width=0.5,
        colorbar=dict(
            title="Total Renewable<br>Generation (GWh)",
            tickformat=","
        ),
        hovertemplate='<b>%{text}</b><br>' +
                      'Renewable Generation: %{z:,.0f} GWh<br>' +
                      '<extra></extra>',
        visible=False
    )
    fig.add_trace(trace)

#Configure the dropdown filter to change the year:
dropdown_buttons = []

#Add the 'Average' option:
dropdown_buttons.append(
    dict(
        label='Average',
        method="update",
        args=[
            {"visible": [True] + [False] * len(years)},
            {"title": "Total Renewable Energy Generation by Country (Average Across All Years)"}
        ]
    )
)

#Adding year specific options to the dropdown:
for i, year in enumerate(years):
    visible = [False] + [False] * len(years)
    visible[i + 1] = True

    dropdown_buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[
                {"visible": visible},
                {"title": f"Total Renewable Energy Generation by Country - {year}"}
            ]
        )
    )

#Update the layout:
fig.update_layout(
    title="Total Renewable Energy Generation by Country (Average Across All Years)",
    geo=dict(
        scope='world',
        projection=dict(type='natural earth'),
        showland=True,
        landcolor='rgb(243, 243, 243)',
        coastlinecolor='rgb(204, 204, 204)',
        showlakes=True,
        lakecolor='rgb(230, 245, 255)',
        showcountries=True,
        countrycolor='rgb(204, 204, 204)',
        countrywidth=0.5
    ),
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.11,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="white",
            bordercolor="gray",
            borderwidth=1
        )
    ],
    annotations=[
        dict(
            text="Select Year:",
            showarrow=False,
            x=0.01,
            y=1.15,
            xref="paper",
            yref="paper",
            align="left",
            font=dict(size=12)
        )
    ],
    height=600,
    margin=dict(l=0, r=0, t=80, b=0)
)

#Display the figure:
fig.show()

### **3. f. Loses - Analysis**
- Analysis for loses reported in the dataset by the Countries
- Calculate Total Losses and Net Electricity Production
- Calculate Percentage of Losses

In [ ]:
#Create intermediate Losses df selecting relevant records:
losses_df = df[
    (df['Country'].isin(core_members)) &
    (df['Balance'].isin(['Distribution Losses', 'Net Electricity Production']))
].copy()

#Introduce Quarters:
losses_df['Quarter'] = losses_df['Time'].dt.to_period('Q')

#Group by the records over Country, Quarter, Balance and unstack the df:
grouped_losses = losses_df.groupby(['Country', 'Quarter', 'Balance'])['Value'].sum().unstack()

#Fill NA with 0:
grouped_losses['Total Losses'] = grouped_losses['Distribution Losses'].fillna(0)
grouped_losses['Net Electricity Production'] = grouped_losses['Net Electricity Production'].fillna(0)

#Calculate Percentage of Losses:
grouped_losses['Percentage of Losses'] = (
    grouped_losses['Total Losses'] / grouped_losses['Net Electricity Production'].replace(0, np.nan) * 100
).fillna(0)

#Select only relevant columns:
df_losses_summary = grouped_losses[['Total Losses', 'Net Electricity Production', 'Percentage of Losses']].reset_index()

#Display the results:
display(df_losses_summary.head())

Balance,Country,Quarter,Total Losses,Net Electricity Production,Percentage of Losses
0,Australia,2010Q1,4099.835,118701.144,3.453914
1,Australia,2010Q2,4043.456,116095.627,3.482867
2,Australia,2010Q3,4282.565,124053.440,3.452194
3,Australia,2010Q4,3954.773,114399.794,3.456976
4,Australia,2011Q1,4093.377,127338.903,3.214553


### **3. g. Loses - Data Visualization**
- Create a pivot table of Country Year and the Percentage of losses
- Visualize the data in a heatmap

In [ ]:
#Define Quarters
df_losses_summary['Year'] = df_losses_summary['Quarter'].astype(str).str[:4].astype(int)

#From the Losses_summary df calculate the Mean of Percentage of Losses:
df_annual_losses = df_losses_summary.groupby(['Country', 'Year']).agg({
    'Percentage of Losses': 'mean'
}).reset_index()

#Pivot df to create a heatmap structure:
heatmap_data = df_annual_losses.pivot(
    index='Country',
    columns='Year',
    values='Percentage of Losses'
)

#Drop Missing Values:
heatmap_data = heatmap_data.dropna(how='any')

heatmap_data = heatmap_data.sort_index()

#Initialize empty list:
annotations = []

#Calculate mean values of all present values:
mean_value = heatmap_data.values[~np.isnan(heatmap_data.values)].mean()

#Loop through each country and year:
for i, country in enumerate(heatmap_data.index):
    for j, year in enumerate(heatmap_data.columns):
        value = heatmap_data.iloc[i, j]
        #Limit to 2 decimal places:
        text = f"{value:.2f}%"
        #Set font color as white if value is high or else black:
        font_color = "white" if value > mean_value else "black"

        #Append annotations details:
        annotations.append(
            dict(
                x=year,
                y=country,
                text=text,
                showarrow=False,
                font=dict(size=10, color=font_color)
            )
        )

#Create plotly heatmap:
fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale=[
        #Low loss rate:
        [0.0, 'rgb(34, 139, 34)'],
        [0.3, 'rgb(144, 238, 144)'],
        #Medium loss rate
        [0.5, 'rgb(255, 255, 0)'],
        [0.7, 'rgb(255, 165, 0)'],
        #High Loss rate:
        [1.0, 'rgb(139, 0, 0)']
    ],
    colorbar=dict(
        title="Distribution<br>Loss Rate (%)",
        titleside="right",
        tickmode="linear",
        tick0=0,
        dtick=2,
        thickness=15,
        len=0.7
    ),
    hovertemplate='<b>%{y}</b><br>' +
                  'Year: %{x}<br>' +
                  'Loss Rate: %{z:.2f}%<br>' +
                  '<extra></extra>',
    zmid=None
))

#Configure layout details:
fig.update_layout(
    title={
        'text': "Electricity Distribution Efficiency: Loss Rates Across OECD Countries (2010-2024)",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 16, 'color': 'rgb(50, 50, 50)'}
    },
    xaxis=dict(
        title="Year",
        side="bottom",
        tickmode='linear',
        tick0=heatmap_data.columns.min(),
        dtick=1,
        showgrid=True,
        gridcolor='white',
        gridwidth=2
    ),
    yaxis=dict(
        title="Country",
        showgrid=True,
        gridcolor='white',
        gridwidth=2,
        tickfont=dict(size=10)
    ),
    width=1400,
    height=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    margin=dict(l=150, r=100, t=100, b=80)
)

#Add cell level annotations to the heatmap
fig.update_layout(annotations=annotations)

#Display the figure
fig.show()

### **3. h. Consumption Growth Ratio - YoY**

####**- Calculate CAGR for Consumption**
- Analysis for growth in Consumption reported for all core countries in the dataset
- Calculate CAGR for Consumption by comparing the Percentage change of Volumes of Consumption a year apart

In [ ]:
#Create an intermediate consumption df:
consumption_df = df[
    (df['Country'].isin(core_members)) &
    (df['Balance'] == 'Final Consumption (Calculated)')
].copy()

#Capture year from Time column:
consumption_df['Year'] = consumption_df['Time'].dt.year

#Group by the value over Country and Year:
yearly_consumption = consumption_df.groupby(['Country', 'Year'])['Value'].sum().reset_index()

#Calculate CAGR by calculating the percentage change for a Country over a year:
yearly_consumption['CAGR'] = yearly_consumption.groupby('Country')['Value'].pct_change() * 100

#Remove missing rows:
yearly_consumption = yearly_consumption.dropna(subset=['CAGR']).reset_index(drop=True)

#Create a new df called CAGR selecting only required columns:
df_cagr = yearly_consumption[['Country', 'Year', 'CAGR']].copy()

#Round CAGR value to 2 decimal places
df_cagr['CAGR'] = df_cagr['CAGR'].round(2)

#Display the results
display(df_cagr.head())

,Country,Year,CAGR
0,Australia,2011,1.53
1,Australia,2012,-0.46
2,Australia,2013,-1.58
3,Australia,2014,3.22
4,Australia,2015,2.45


#### **- Standard Deviation for Consumption**
- Use the CAGR calculated in previous cell to calculate the Standard Deviation of Yearly Consumption

In [ ]:
#Overwrite the intermediate consumption df:
consumption_df = df[
    (df['Country'].isin(core_members)) &
    (df['Balance'] == 'Final Consumption (Calculated)')
].copy()

#Capture year from Time column:
consumption_df['Year'] = consumption_df['Time'].dt.year

#Aggregate total consumption values:
yearly_consumption = consumption_df.groupby(['Country', 'Year'])['Value'].sum().reset_index()

#Calculate mean and standard deviation of consumption values:
stdDev_summary = yearly_consumption.groupby('Country')['Value'].agg(['mean', 'std']).reset_index()

#Rename columns:
stdDev_summary = stdDev_summary.rename(columns={'mean': 'Mean Yearly Consumption', 'std': 'Standard Deviation of Yearly Consumption'})

#Calculate Std Dev % by dividing std over mean and multiplying by 100 calculated in the previous steps:
stdDev_summary['Standard Deviation (%)'] = (stdDev_summary['Standard Deviation of Yearly Consumption'] / stdDev_summary['Mean Yearly Consumption']) * 100

#Round off all values:
stdDev_summary['Mean Yearly Consumption'] = stdDev_summary['Mean Yearly Consumption'].round(2)
stdDev_summary['Standard Deviation of Yearly Consumption'] = stdDev_summary['Standard Deviation of Yearly Consumption'].round(2)
stdDev_summary['Standard Deviation (%)'] = stdDev_summary['Standard Deviation (%)'].round(2)

#Select only relevant columns in a new df:
df_stdDev = stdDev_summary[['Country', 'Mean Yearly Consumption', 'Standard Deviation of Yearly Consumption', 'Standard Deviation (%)']]

#Display the results:
display(df_stdDev.head())

,Country,Mean Yearly Consumption,Standard Deviation of Yearly Consumption,Standard Deviation (%)
0,Australia,234602.70,11376.69,4.85
1,Austria,64446.48,1700.04,2.64
2,Belgium,82626.71,2224.08,2.69
3,Canada,544603.78,17157.53,3.15
4,Chile,72183.83,9462.89,13.11


####**- Votality - Analysis for Consumption**
- Merge the CAGR Consumption with Standard Deviation for Consumption
- Use the merged dataframe to calculate Volatility

In [ ]:
#Create an intermediate df by merging the necessary columns in CAGR and Std Dev dataframes:
df_volatility_combined = pd.merge(df_cagr, df_stdDev[['Country', 'Standard Deviation (%)']], on='Country')

#Calculate Volatility by Subtracting the Std Dev value from CAGR and taking the square to ensure only positive values are in our output:
df_volatility_combined['Calculated Volatility'] = (df_volatility_combined['CAGR'] - df_volatility_combined['Standard Deviation (%)'])**2

#Select relevant columns in final df:
df_combined_analysis = df_volatility_combined[['Country', 'Year', 'CAGR', 'Standard Deviation (%)', 'Calculated Volatility']]
#Display the results:
display(df_combined_analysis.head())

,Country,Year,CAGR,Standard Deviation (%),Calculated Volatility
0,Australia,2011,1.53,4.85,11.0224
1,Australia,2012,-0.46,4.85,28.1961
2,Australia,2013,-1.58,4.85,41.3449
3,Australia,2014,3.22,4.85,2.6569
4,Australia,2015,2.45,4.85,5.7600


####**- Volatility of Consumption Visualization**
- Quadrant axes are defined taking the average values for CAGR and Volatility values of Consumption
- Visualize the values for countries in a scatter plot

In [ ]:
df_normalized = df_combined_analysis.copy()

#Initalize MinMaxScalers for CAGR and Volatility:
scaler_cagr = MinMaxScaler()
scaler_volatility = MinMaxScaler()

#Apply min max scaling to normalize CAGR and Volatility Values:
df_normalized['CAGR_Normalized'] = scaler_cagr.fit_transform(df_normalized[['CAGR']])
#Store normalized values to new column in df:
df_normalized['Volatility_Normalized'] = scaler_volatility.fit_transform(df_normalized[['Calculated Volatility']])

#Calculate the Mean Normalized CAGR and Volatility for all countries:
df_average = df_normalized.groupby('Country').agg({
    'CAGR_Normalized': 'mean',
    'Volatility_Normalized': 'mean'
}).reset_index()

#Calculate Median Values of CAGR and Volatility to be used as Benchmark values for this visualization:
FIXED_MEDIAN_GROWTH = df_average['CAGR_Normalized'].median()
FIXED_MEDIAN_VOLATILITY = df_average['Volatility_Normalized'].median()

print(f"Fixed Median Growth (normalized): {FIXED_MEDIAN_GROWTH:.4f}")
print(f"Fixed Median Volatility (normalized): {FIXED_MEDIAN_VOLATILITY:.4f}")

#Sort by Years
years = sorted(df_normalized['Year'].unique())
year_options = ['Average'] + [str(year) for year in years]

#Initialize plotly figure
fig = go.Figure()

#Configure the scatter plot:
fig.add_trace(
    go.Scatter(
        x=df_average['CAGR_Normalized'],
        y=df_average['Volatility_Normalized'],
        mode='markers',
        marker=dict(size=12, color='#3b82f6', opacity=0.7, line=dict(width=1, color='white')),
        text=df_average['Country'],
        hovertemplate='<b>%{text}</b><br>Normalized Growth: %{x:.3f}<br>Normalized Volatility: %{y:.3f}<extra></extra>',
        name='Countries',
        visible=True
    )
)

#Loop through each year:
for year in years:
    df_year = df_normalized[df_normalized['Year'] == year]
    fig.add_trace(
        go.Scatter(
            x=df_year['CAGR_Normalized'],
            y=df_year['Volatility_Normalized'],
            mode='markers',
            marker=dict(size=12, color='#3b82f6', opacity=0.7, line=dict(width=1, color='white')),
            text=df_year['Country'],
            hovertemplate='<b>%{text}</b><br>Normalized Growth: %{x:.3f}<br>Normalized Volatility: %{y:.3f}<extra></extra>',
            name='Countries',
            visible=False
        )
    )

#Initialize array of buttons to be used later:
buttons = []

#Add the fixed benchmark axes:
fixed_shapes = [
    dict(type='line', x0=FIXED_MEDIAN_GROWTH, x1=FIXED_MEDIAN_GROWTH,
         y0=0, y1=1, yref='paper', line=dict(dash='dash', color='gray', width=2)),
    dict(type='line', y0=FIXED_MEDIAN_VOLATILITY, y1=FIXED_MEDIAN_VOLATILITY,
         x0=0, x1=1, xref='paper', line=dict(dash='dash', color='gray', width=2))
]

#Append the years and 'Average' to the buttons array:
buttons.append(
    dict(
        label='Average',
        method='update',
        args=[
            {'visible': [True] + [False] * len(years)},
            {'shapes': fixed_shapes}
        ]
    )
)

#Configure the dropdown:
for i, year in enumerate(years):
    visible = [False] * (len(years) + 1)
    visible[i + 1] = True

    buttons.append(
        dict(
            label=str(year),
            method='update',
            args=[
                {'visible': visible},
                {'shapes': fixed_shapes}
            ]
        )
    )

#Add labels and other details to the scatter plot. Define Quadrants:
fig.update_layout(
    title={
        'text': 'Growth vs Volatility Quadrant Analysis (Normalized)<br><sub>Quadrants based on overall median values across all years</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#1f2937'}
    },
    xaxis_title='Normalized Growth (CAGR)',
    yaxis_title='Normalized Volatility',
    showlegend=False,
    hovermode='closest',
    width=1000,
    height=700,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='#e5e7eb',
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='#9ca3af',
        range=[-0.05, 1.05]
    ),
    yaxis=dict(
        showgrid=True,
        gridwidth=1,
        gridcolor='#e5e7eb',
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='#9ca3af',
        range=[-0.05, 1.05]
    ),
    updatemenus=[
        dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=0.17,
            xanchor='left',
            y=1.15,
            yanchor='top',
            bgcolor='white',
            bordercolor='#d1d5db',
            borderwidth=1
        )
    ],
    shapes=fixed_shapes,
    annotations=[
        dict(
            text='Select Year:',
            x=0.02,
            y=1.15,
            xref='paper',
            yref='paper',
            showarrow=False,
            font=dict(size=12, color='#374151'),
            xanchor='left'
        ),
        dict(text='Q1: High Growth<br>High Volatility', x=0.95, y=0.95, xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color='#6b7280'), xanchor='right', yanchor='top'),
        dict(text='Q2: Low Growth<br>High Volatility', x=0.05, y=0.95, xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color='#6b7280'), xanchor='left', yanchor='top'),
        dict(text='Q3: Low Growth<br>Low Volatility', x=0.05, y=0.05, xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color='#6b7280'), xanchor='left', yanchor='bottom'),
        dict(text='Q4: High Growth<br>Low Volatility', x=0.95, y=0.05, xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color='#6b7280'), xanchor='right', yanchor='bottom'),
    ]
)

#Show the figure
fig.show()

#Function to classify quadrants to categorise countries:
def classify_quadrant(growth, volatility):
    """Classify country into quadrant based on fixed median thresholds"""
    if growth >= FIXED_MEDIAN_GROWTH and volatility >= FIXED_MEDIAN_VOLATILITY:
        return "Q1: High Growth, High Volatility"
    elif growth < FIXED_MEDIAN_GROWTH and volatility >= FIXED_MEDIAN_VOLATILITY:
        return "Q2: Low Growth, High Volatility"
    elif growth < FIXED_MEDIAN_GROWTH and volatility < FIXED_MEDIAN_VOLATILITY:
        return "Q3: Low Growth, Low Volatility"
    else:
        return "Q4: High Growth, Low Volatility"

df_average['Quadrant'] = df_average.apply(
    lambda row: classify_quadrant(row['CAGR_Normalized'], row['Volatility_Normalized']),
    axis=1
)

#Print the results:
print("\n" + "="*60)
print("COUNTRY CLASSIFICATION BY QUADRANT (Based on Average Data)")
print("="*60)
for quadrant in sorted(df_average['Quadrant'].unique()):
    countries = df_average[df_average['Quadrant'] == quadrant]['Country'].tolist()
    print(f"\n{quadrant}:")
    print(f"  Countries ({len(countries)}): {', '.join(countries)}")

print("\n" + "="*60)
print("NORMALIZED DATA SAMPLE:")
print("="*60)
display(df_normalized[['Country', 'Year', 'CAGR', 'CAGR_Normalized', 'Calculated Volatility', 'Volatility_Normalized']].head(10))

print("\n" + "="*60)
print("AVERAGE DATA WITH QUADRANT CLASSIFICATION:")
print("="*60)
display(df_average.sort_values('Quadrant'))

Fixed Median Growth (normalized): 0.3044
Fixed Median Volatility (normalized): 0.0690



COUNTRY CLASSIFICATION BY QUADRANT (Based on Average Data)

Q1: High Growth, High Volatility:
  Countries (11): Chile, Colombia, Hungary, Iceland, Ireland, Korea, Lithuania, Mexico, Poland, Republic of Turkiye, Slovenia

Q2: Low Growth, High Volatility:
  Countries (8): Estonia, Finland, France, Germany, Greece, Japan, Slovak Republic, United Kingdom

Q3: Low Growth, Low Volatility:
  Countries (10): Belgium, Czech Republic, Italy, Luxembourg, Netherlands, New Zealand, Portugal, Spain, Sweden, Switzerland

Q4: High Growth, Low Volatility:
  Countries (8): Australia, Austria, Canada, Costa Rica, Denmark, Latvia, Norway, United States

NORMALIZED DATA SAMPLE:


,Country,Year,CAGR,CAGR_Normalized,Calculated Volatility,Volatility_Normalized
0,Australia,2011,1.53,0.343968,11.0224,0.035101
1,Australia,2012,-0.46,0.283500,28.1961,0.089794
2,Australia,2013,-1.58,0.249468,41.3449,0.131670
3,Australia,2014,3.22,0.395321,2.6569,0.008459
4,Australia,2015,2.45,0.371923,5.7600,0.018341
5,Australia,2016,0.56,0.314494,18.4041,0.058609
6,Australia,2017,0.87,0.323914,15.8404,0.050445
7,Australia,2018,0.39,0.309328,19.8916,0.063347
8,Australia,2019,1.92,0.355819,8.5849,0.027338
9,Australia,2020,-0.06,0.295655,24.1081,0.076775



AVERAGE DATA WITH QUADRANT CLASSIFICATION:


,Country,CAGR_Normalized,Volatility_Normalized,Quadrant
14,Hungary,0.344142,0.127257,"Q1: High Growth, High Volatility"
29,Republic of Turkiye,0.425837,0.603248,"Q1: High Growth, High Volatility"
23,Mexico,0.384707,0.235568,"Q1: High Growth, High Volatility"
21,Lithuania,0.386270,0.303279,"Q1: High Growth, High Volatility"
4,Chile,0.392043,0.340903,"Q1: High Growth, High Volatility"
5,Colombia,0.387694,0.123583,"Q1: High Growth, High Volatility"
19,Korea,0.339693,0.074617,"Q1: High Growth, High Volatility"
31,Slovenia,0.318488,0.071426,"Q1: High Growth, High Volatility"
16,Ireland,0.356644,0.212617,"Q1: High Growth, High Volatility"
15,Iceland,0.329709,0.083825,"Q1: High Growth, High Volatility"


###**3. i. Imports Growth Rate - YoY**

####**- Calculate CAGR for Imports**
- Analysis for growth in Imports reported for all core countries in the dataset
- Calculate CAGR for Imports

In [ ]:
#Select the total imports for Core Countries:
imports_df = df[
    (df['Country'].isin(core_members)) &
    (df['Balance'] == 'Total Imports')
].copy()

imports_df['Year'] = imports_df['Time'].dt.year

#Aggregate Imports over Country and Year:
yearly_imports = imports_df.groupby(['Country', 'Year'])['Value'].sum().reset_index()

#Calculate Percentage change of CAGR for a Country over a year:
yearly_imports['CAGR'] = yearly_imports.groupby('Country')['Value'].pct_change() * 100

#Drop Null values:
yearly_imports = yearly_imports.dropna(subset=['CAGR']).reset_index(drop=True)

#Copy relevant columns to new df:
df_cagr_imports = yearly_imports[['Country', 'Year', 'CAGR']].copy()

#Round the CAGR value to 2 decimal places:
df_cagr_imports['CAGR'] = df_cagr_imports['CAGR'].round(2)

#Display the final reults:
display(df_cagr_imports.head())

,Country,Year,CAGR
0,Austria,2011,25.46
1,Austria,2012,-6.19
2,Austria,2013,6.53
3,Austria,2014,7.02
4,Austria,2015,10.02


#### **- Standard Deviation for Import**
- Calculate Standard Deviation of Imports

In [ ]:
#Overwrite the data in the imports df:
imports_df = df[
    (df['Country'].isin(core_members)) &
    (df['Balance'] == 'Total Imports')
].copy()

imports_df['Year'] = imports_df['Time'].dt.year

#Aggregate Imports over Country and Year:
yearly_imports = imports_df.groupby(['Country', 'Year'])['Value'].sum().reset_index()

#Calculate mean and standard deviation of import values:
stdDev_summary_imports = yearly_imports.groupby('Country')['Value'].agg(['mean', 'std']).reset_index()

#Rename columns:
stdDev_summary_imports = stdDev_summary_imports.rename(columns={'mean': 'Mean Yearly Imports', 'std': 'Standard Deviation of Yearly Imports'})

#Calculate Std Dev % by dividing std over mean and multiplying by 100 calculated in the previous steps:
stdDev_summary_imports['Standard Deviation (%)'] = (stdDev_summary_imports['Standard Deviation of Yearly Imports'] / stdDev_summary_imports['Mean Yearly Imports']) * 100

#Round off all values:
stdDev_summary_imports['Mean Yearly Imports'] = stdDev_summary_imports['Mean Yearly Imports'].round(2)
stdDev_summary_imports['Standard Deviation of Yearly Imports'] = stdDev_summary_imports['Standard Deviation of Yearly Imports'].round(2)
stdDev_summary_imports['Standard Deviation (%)'] = stdDev_summary_imports['Standard Deviation (%)'].round(2)

#Select only relevant columns in a new df:
df_stdDev_imports = stdDev_summary_imports[['Country', 'Mean Yearly Imports', 'Standard Deviation of Yearly Imports', 'Standard Deviation (%)']]
#Display the results:
display(df_stdDev_imports.head())

,Country,Mean Yearly Imports,Standard Deviation of Yearly Imports,Standard Deviation (%)
0,Australia,0.00,0.00,NaN
1,Austria,25283.62,3216.92,12.72
2,Belgium,17600.70,4991.94,28.36
3,Canada,13566.63,4379.30,32.28
4,Chile,211.25,395.80,187.36


####**- Volatility - Analysis for Imports**
- Merge CAGR and Standard Deviation dataframes
- Calculate Volatility of Imports

In [ ]:
#Create an intermediate df by merging the necessary columns in CAGR and Std Dev dataframes:
df_volatility_combined_imports = pd.merge(df_cagr_imports, df_stdDev_imports[['Country', 'Standard Deviation (%)']], on='Country')

#Calculate Volatility by Subtracting the Std Dev value from CAGR and taking the square to ensure only positive values are in our output:
df_volatility_combined_imports['Calculated Volatility'] = (df_volatility_combined_imports['CAGR'] - df_volatility_combined_imports['Standard Deviation (%)'])**2

#Select relevant columns in final df:
df_combined_analysis_imports = df_volatility_combined_imports[['Country', 'Year', 'CAGR', 'Standard Deviation (%)', 'Calculated Volatility']]

#Display the final results:
display(df_combined_analysis_imports.head())

,Country,Year,CAGR,Standard Deviation (%),Calculated Volatility
0,Austria,2011,25.46,12.72,162.3076
1,Austria,2012,-6.19,12.72,357.5881
2,Austria,2013,6.53,12.72,38.3161
3,Austria,2014,7.02,12.72,32.4900
4,Austria,2015,10.02,12.72,7.2900


### **3. j. Exports Growth Rate - YoY**

####**- Calculate CAGR for Exports**
- Analysis for growth in Exports reported for all core countries in the dataset
- Calculate CAGR for Exports

In [ ]:
#Select the total exports for Core Countries:
exports_df = df[
    (df['Country'].isin(core_members)) &
    (df['Balance'] == 'Total Exports')
].copy()

exports_df['Year'] = exports_df['Time'].dt.year

#Aggregate Exports over Country and Year:
yearly_exports = exports_df.groupby(['Country', 'Year'])['Value'].sum().reset_index()

#Calculate Percentage change of CAGR for a Country over a year:
yearly_exports['CAGR'] = yearly_exports.groupby('Country')['Value'].pct_change() * 100

#Drop Null values:
yearly_exports = yearly_exports.dropna(subset=['CAGR']).reset_index(drop=True)

#Copy relevant columns to new df:
df_cagr_exports = yearly_exports[['Country', 'Year', 'CAGR']].copy()

#Round the CAGR value to 2 decimal places:
df_cagr_exports['CAGR'] = df_cagr_exports['CAGR'].round(2)

#Display the final reults:
display(df_cagr_exports.head())

,Country,Year,CAGR
0,Austria,2011,-3.98
1,Austria,2012,22.94
2,Austria,2013,-14.24
3,Austria,2014,-1.43
4,Austria,2015,10.84


####**- Standard Deviation - Analysis for Exports**
- Calculate Standard Deviation for Exports

In [ ]:
#Select the total exports for Core Countries:
exports_df = df[
    (df['Country'].isin(core_members)) &
    (df['Balance'] == 'Total Exports')
].copy()

exports_df['Year'] = exports_df['Time'].dt.year

#Aggregate Exports over Country and Year:
yearly_exports = exports_df.groupby(['Country', 'Year'])['Value'].sum().reset_index()

#Calculate mean and standard deviation of export values:
stdDev_summary_exports = yearly_exports.groupby('Country')['Value'].agg(['mean', 'std']).reset_index()

#Rename columns:
stdDev_summary_exports = stdDev_summary_exports.rename(columns={'mean': 'Mean Yearly Exports', 'std': 'Standard Deviation of Yearly Exports'})

#Calculate Std Dev % by dividing std over mean and multiplying by 100 calculated in the previous steps:
stdDev_summary_exports['Standard Deviation (%)'] = (stdDev_summary_exports['Standard Deviation of Yearly Exports'] / stdDev_summary_exports['Mean Yearly Exports']) * 100

#Round off all values:
stdDev_summary_exports['Mean Yearly Exports'] = stdDev_summary_exports['Mean Yearly Exports'].round(2)
stdDev_summary_exports['Standard Deviation of Yearly Exports'] = stdDev_summary_exports['Standard Deviation of Yearly Exports'].round(2)
stdDev_summary_exports['Standard Deviation (%)'] = stdDev_summary_exports['Standard Deviation (%)'].round(2)

#Select only relevant columns in a new df:
df_stdDev_exports = stdDev_summary_exports[['Country', 'Mean Yearly Exports', 'Standard Deviation of Yearly Exports', 'Standard Deviation (%)']]
#Display the results:
display(df_stdDev_exports.head())

,Country,Mean Yearly Exports,Standard Deviation of Yearly Exports,Standard Deviation (%)
0,Australia,0.00,0.00,NaN
1,Austria,20122.38,2520.55,12.53
2,Belgium,11782.40,6710.22,56.95
3,Canada,59274.71,10323.07,17.42
4,Chile,6.91,15.97,231.06


####**- Volatility - Analysis for Exports**
- Merge the CAGR and Standard Deviation dataframes
- Calculate volatility for Exports

In [ ]:
#Create an intermediate df by merging the necessary columns in CAGR and Std Dev dataframes:
df_volatility_combined_exports = pd.merge(df_cagr_exports, df_stdDev_exports[['Country', 'Standard Deviation (%)']], on='Country')

#Calculate Volatility by Subtracting the Std Dev value from CAGR and taking the square to ensure only positive values are in our output:
df_volatility_combined_exports['Calculated Volatility'] = (df_volatility_combined_exports['CAGR'] - df_volatility_combined_exports['Standard Deviation (%)'])**2

#Select relevant columns in final df:
df_combined_analysis_exports = df_volatility_combined_exports[['Country', 'Year', 'CAGR', 'Standard Deviation (%)', 'Calculated Volatility']]
#Display the final results:
display(df_combined_analysis_exports.head())

,Country,Year,CAGR,Standard Deviation (%),Calculated Volatility
0,Austria,2011,-3.98,12.53,272.5801
1,Austria,2012,22.94,12.53,108.3681
2,Austria,2013,-14.24,12.53,716.6329
3,Austria,2014,-1.43,12.53,194.8816
4,Austria,2015,10.84,12.53,2.8561


###**3. k. Net Ratio QoQ - Analysis for Export**
- Calculate Net Export Ratio by Taking the difference of Total Imports and Exports and divide by Net Electricity Production

In [ ]:
#Capture relevant columns for Export Ratio Calculation:
net_export_df = df[
    (df['Country'].isin(core_members)) &
    df['Balance'].isin([
    'Total Exports',
    'Total Imports',
    'Net Electricity Production'
])].copy()

#Define Quarters:
net_export_df['Quarter'] = net_export_df['Time'].dt.to_period('Q')

display(net_export_df.head())

#Pivot net_export_df by Country and Quarter, summing 'Value' for each 'Balance':
net_export_pivot = net_export_df.pivot_table(
    index=['Country', 'Quarter'],
    columns='Balance',
    values='Value',
    aggfunc='sum'
).reset_index()

display(net_export_pivot.head())

#Calculate Net Export Ratio by taking the difference of the Imports and Exports and Dividing by Net production:
net_export_pivot['Net Export Ratio'] = (
    (net_export_pivot['Total Exports'].fillna(0) - net_export_pivot['Total Imports'].fillna(0)) /
    net_export_pivot['Net Electricity Production'].replace(0, np.nan)
)

#Fill NA values with 0:
net_export_pivot['Net Export Ratio'] = net_export_pivot['Net Export Ratio'].fillna(0)

#Copy relevant columns to new df:
df_net_export_ratio = net_export_pivot[['Country', 'Quarter', 'Net Export Ratio']]
#Display the results:
display(df_net_export_ratio.head())

,Country,Time,Balance,Product,Value,Unit,Group,Energy Type,Quarter
0,Australia,2024-12-01,Net Electricity Production,Electricity,24109.2205,GWh,Core Member,Unspecified,2024Q4
1,Australia,2024-12-01,Net Electricity Production,Hydro,969.5320,GWh,Core Member,Renewable,2024Q4
2,Australia,2024-12-01,Net Electricity Production,"Coal, Peat and Manufactured Gases",9554.5578,GWh,Core Member,Non-Renewable,2024Q4
3,Australia,2024-12-01,Net Electricity Production,Oil and Petroleum Products,350.3522,GWh,Core Member,Non-Renewable,2024Q4
4,Australia,2024-12-01,Net Electricity Production,Natural Gas,3519.4580,GWh,Core Member,Non-Renewable,2024Q4


Balance,Country,Quarter,Net Electricity Production,Total Exports,Total Imports
0,Australia,2010Q1,118701.144,NaN,NaN
1,Australia,2010Q2,116095.627,NaN,NaN
2,Australia,2010Q3,124053.440,NaN,NaN
3,Australia,2010Q4,114399.794,NaN,NaN
4,Australia,2011Q1,127338.903,NaN,NaN


Balance,Country,Quarter,Net Export Ratio
0,Australia,2010Q1,0.0
1,Australia,2010Q2,0.0
2,Australia,2010Q3,0.0
3,Australia,2010Q4,0.0
4,Australia,2011Q1,0.0


####**- Visualization for Net Export Ratio**
- Visualize Net Export Ratio as a bar chart

In [ ]:
df_net_export_ratio = df_net_export_ratio.copy()
#Extract the year from the 'Quarter'
df_net_export_ratio['Year'] = df_net_export_ratio['Quarter'].astype(str).str[:4].astype(int)

#Take the mean of the Net Export Ratio by grouping the records over Country and Year
df_avg_export = df_net_export_ratio.groupby(['Country', 'Year']).agg({
    'Net Export Ratio': 'mean'
}).reset_index()

#Calculate the average Net Export Ratio for each country:
df_avg_export_all = df_net_export_ratio.groupby('Country').agg({'Net Export Ratio': 'mean'}).reset_index()

#Filter datapoints with not null values:
df_avg_export = df_avg_export[df_avg_export['Net Export Ratio'] != 0]
df_avg_export_all = df_avg_export_all[df_avg_export_all['Net Export Ratio'] != 0]

#Sort the df over year:
years = sorted(df_avg_export['Year'].unique())

#Initialize an empty Plotly figure:
fig = go.Figure()

#Sort the average net import data by country and assign colors based on whether the Net Import Ratio values:
avg_data = df_avg_export_all.sort_values('Country')
colors_avg = ['green' if val >= 0 else 'red' for val in avg_data['Net Export Ratio']]

#Create a bar trace for average Net Import Ratio:
trace_avg = go.Bar(
    x=avg_data['Country'],
    y=avg_data['Net Export Ratio'],
    marker=dict(
        color=colors_avg,
        line=dict(color='darkgray', width=0.5)
    ),
    text=[f"{val:.3f}" for val in avg_data['Net Export Ratio']],
    textposition='outside',
    textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>' +
                  'Net Export Ratio: %{y:.4f}<br>' +
                  '<extra></extra>',
    visible=True
)
fig.add_trace(trace_avg)

#Create a bar trace for each year's Net Import Ratio:
for year in years:
    year_data = df_avg_export[df_avg_export['Year'] == year].copy()

    year_data = year_data.sort_values('Country')

    colors = ['green' if val >= 0 else 'red' for val in year_data['Net Export Ratio']]

    trace = go.Bar(
        x=year_data['Country'],
        y=year_data['Net Export Ratio'],
        marker=dict(
            color=colors,
            line=dict(color='darkgray', width=0.5)
        ),
        text=[f"{val:.3f}" for val in year_data['Net Export Ratio']],
        textposition='outside',
        textfont=dict(size=9),
        hovertemplate='<b>%{x}</b><br>' +
                      'Net Export Ratio: %{y:.4f}<br>' +
                      '<extra></extra>',
        visible=(year == years[0])
    )
    fig.add_trace(trace)

#Initialize dropdown button array:
dropdown_buttons = []

#Set up dropdown buttons to toggle visibility of the bar traces:
dropdown_buttons.append(
    dict(
        label='Average',
        method="update",
        args=[
            {"visible": [True] + [False] * len(years)},
            {"title": "Average Net Export Ratio by Country (All Years)"}
        ]
    )
)

for i, year in enumerate(years):
    visible = [False] + [False] * len(years)
    visible[i + 1] = True

    dropdown_buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[
                {"visible": visible},
                {"title": f"Average Net Export Ratio by Country - {year}"}
            ]
        )
    )

#Configure the layout:
fig.update_layout(
    title={
        'text': f"Average Net Export Ratio by Country - {years[0]}",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 16, 'color': 'rgb(50, 50, 50)'}
    },
    xaxis=dict(
        title="Country",
        tickangle=-45,
        showgrid=False,
        tickfont=dict(size=10)
    ),
    yaxis=dict(
        title="Net Export Ratio",
        showgrid=True,
        gridcolor='lightgray',
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black'
    ),
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.12,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="white",
            bordercolor="gray",
            borderwidth=1
        )
    ],
    annotations=[
        dict(
            text="Select Year:",
            showarrow=False,
            x=0.01,
            y=1.15,
            xref="paper",
            yref="paper",
            align="left",
            font=dict(size=12)
        )
    ],
    showlegend=False,
    height=600,
    width=1400,
    plot_bgcolor='white',
    margin=dict(l=80, r=80, t=120, b=120)
)

#Display the results
fig.show()

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Total countries analyzed: {df_avg_export['Country'].nunique()}")
print(f"Year range: {df_avg_export['Year'].min()} - {df_avg_export['Year'].max()}")
print(f"\nAverage Net Export Ratio across all countries and years: {df_avg_export['Net Export Ratio'].mean():.4f}")
print(f"Median Net Export Ratio: {df_avg_export['Net Export Ratio'].median():.4f}")
print(f"Min Net Export Ratio: {df_avg_export['Net Export Ratio'].min():.4f}")
print(f"Max Net Export Ratio: {df_avg_export['Net Export Ratio'].max():.4f}")

print("\n" + "="*60)
print("TOP 5 NET Exporters (Average across all years)")
print("="*60)
top_exporters = df_avg_export.groupby('Country')['Net Export Ratio'].mean().sort_values(ascending=False).head(5)
for country, ratio in top_exporters.items():
    print(f"{country}: {ratio:.4f}")

print("\n" + "="*60)
print("TOP 5 NET EXPORTERS (Average across all years)")
print("="*60)
top_exporters = df_avg_export.groupby('Country')['Net Export Ratio'].mean().sort_values(ascending=True).head(5)
for country, ratio in top_exporters.items():
    print(f"{country}: {ratio:.4f}")


SUMMARY STATISTICS
Total countries analyzed: 32
Year range: 2010 - 2024

Average Net Export Ratio across all countries and years: -0.0856
Median Net Export Ratio: -0.0081
Min Net Export Ratio: -1.6080
Max Net Export Ratio: 0.1538

TOP 5 NET Exporters (Average across all years)
Czech Republic: 0.0834
Sweden: 0.0609
France: 0.0459
Norway: 0.0403
Canada: 0.0366

TOP 5 NET EXPORTERS (Average across all years)
Luxembourg: -1.0998
Lithuania: -0.8768
Hungary: -0.1839
Latvia: -0.1727
Finland: -0.1116


###**3. l. Net Ratio QoQ - Analysis for Import**
- Calculate the Net Import Ratio by taking a difference of Net Imports and Exports and dividing by Final Consumption

In [ ]:
#Capture relevant columns for Import Ratio Calculation:
net_import_df = df[
    (df['Country'].isin(core_members)) &
    df['Balance'].isin([
    'Total Exports',
    'Total Imports',
    'Final Consumption (Calculated)'
])].copy()

#Define Quarters:
net_import_df['Quarter'] = net_import_df['Time'].dt.to_period('Q')

display(net_import_df.head())

#Pivot net_import_df by Country and Quarter, summing 'Value' for each 'Balance':
net_import_pivot = net_import_df.pivot_table(
    index=['Country', 'Quarter'],
    columns='Balance',
    values='Value',
    aggfunc='sum'
).reset_index()

display(net_import_pivot.head())

#Calculate Net Import Ratio by taking the difference of the Imports and Exports and Dividing by Final consumption:
net_import_pivot['Net Import Ratio'] = (
    (net_import_pivot['Total Imports'].fillna(0) - net_import_pivot['Total Exports'].fillna(0)) /
    net_import_pivot['Final Consumption (Calculated)'].replace(0, np.nan)
)

#Fill NA values with 0:
net_import_pivot['Net Import Ratio'] = net_import_pivot['Net Import Ratio'].fillna(0)

#Copy relevant columns to new df:
df_net_import_ratio = net_import_pivot[['Country', 'Quarter', 'Net Import Ratio']].copy()
#Display the results:
display(df_net_import_ratio.head())

,Country,Time,Balance,Product,Value,Unit,Group,Energy Type,Quarter
9,Australia,2024-12-01,Final Consumption (Calculated),Electricity,22853.8202,GWh,Core Member,Unspecified,2024Q4
21,Austria,2024-12-01,Total Imports,Electricity,2719.5360,GWh,Core Member,Unspecified,2024Q4
22,Austria,2024-12-01,Total Exports,Electricity,2078.8699,GWh,Core Member,Unspecified,2024Q4
25,Austria,2024-12-01,Final Consumption (Calculated),Electricity,6111.4243,GWh,Core Member,Unspecified,2024Q4
37,Belgium,2024-12-01,Total Imports,Electricity,2391.5483,GWh,Core Member,Unspecified,2024Q4


Balance,Country,Quarter,Final Consumption (Calculated),Total Exports,Total Imports
0,Australia,2010Q1,55230.487,NaN,NaN
1,Australia,2010Q2,53984.108,NaN,NaN
2,Australia,2010Q3,57687.991,NaN,NaN
3,Australia,2010Q4,53188.443,NaN,NaN
4,Australia,2011Q1,59537.443,NaN,NaN


Balance,Country,Quarter,Net Import Ratio
0,Australia,2010Q1,0.0
1,Australia,2010Q2,0.0
2,Australia,2010Q3,0.0
3,Australia,2010Q4,0.0
4,Australia,2011Q1,0.0


####**- Visualization for Net Import Ratio**

In [ ]:
#Extract the year from the 'Quarter'
df_net_import_ratio = df_net_import_ratio.copy()
df_net_import_ratio['Year'] = df_net_import_ratio['Quarter'].astype(str).str[:4].astype(int).copy()

#Take the mean of the Net Import Ratio by grouping the records over Country and Year
df_avg_import = df_net_import_ratio.groupby(['Country', 'Year']).agg({
    'Net Import Ratio': 'mean'
}).reset_index()

#Calculate the average Net Import Ratio for each country:
df_avg_import_all = df_net_import_ratio.groupby('Country').agg({'Net Import Ratio': 'mean'}).reset_index().copy()

#Filter datapoints with not null values:
df_avg_import = df_avg_import[df_avg_import['Net Import Ratio'] != 0]
df_avg_import_all = df_avg_import_all[df_avg_import_all['Net Import Ratio'] != 0]

#Sort the df over year:
years = sorted(df_avg_import['Year'].unique())

#Initialize an empty Plotly figure:
fig = go.Figure()

#Sort the average net import data by country and assign colors based on whether the Net Import Ratio values:
avg_data = df_avg_import_all.sort_values('Country')
colors_avg = ['green' if val >= 0 else 'red' for val in avg_data['Net Import Ratio']]

#Create a bar trace for average Net Import Ratio:
trace_avg = go.Bar(
    x=avg_data['Country'],
    y=avg_data['Net Import Ratio'],
    marker=dict(
        color=colors_avg,
        line=dict(color='darkgray', width=0.5)
    ),
    text=[f"{val:.3f}" for val in avg_data['Net Import Ratio']],
    textposition='outside',
    textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>' +
                  'Net Import Ratio: %{y:.4f}<br>' +
                  '<extra></extra>',
    visible=True
)
fig.add_trace(trace_avg)

#Create a bar trace for each year's Net Import Ratio:
for year in years:
    year_data = df_avg_import[df_avg_import['Year'] == year].copy()

    year_data = year_data.sort_values('Country')

    colors = ['green' if val >= 0 else 'red' for val in year_data['Net Import Ratio']]

    trace = go.Bar(
        x=year_data['Country'],
        y=year_data['Net Import Ratio'],
        marker=dict(
            color=colors,
            line=dict(color='darkgray', width=0.5)
        ),
        text=[f"{val:.3f}" for val in year_data['Net Import Ratio']],
        textposition='outside',
        textfont=dict(size=9),
        hovertemplate='<b>%{x}</b><br>' +
                      'Net Import Ratio: %{y:.4f}<br>' +
                      '<extra></extra>',
        visible=(year == years[0])
    )
    fig.add_trace(trace)

#Initialize dropdown button array:
dropdown_buttons = []

#Set up dropdown buttons to toggle visibility of the bar traces:
for i, year in enumerate(years):
    visible = [False] * len(years)
    visible[i] = True

dropdown_buttons.append(
    dict(
        label='Average',
        method="update",
        args=[
            {"visible": [True] + [False] * len(years)},
            {"title": "Average Net Import Ratio by Country (All Years)"}
        ]
    )
)

for i, year in enumerate(years):
    visible = [False] + [False] * len(years)
    visible[i + 1] = True

    dropdown_buttons.append(
        dict(
            label=str(year),
            method="update",
            args=[
                {"visible": visible},
                {"title": f"Average Net Import Ratio by Country - {year}"}
            ]
        )
    )

#Configure the layout:
fig.update_layout(
    title={
        'text': f"Average Net Import Ratio by Country - {years[0]}",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 16, 'color': 'rgb(50, 50, 50)'}
    },
    xaxis=dict(
        title="Country",
        tickangle=-45,
        showgrid=False,
        tickfont=dict(size=10)
    ),
    yaxis=dict(
        title="Net Import Ratio",
        showgrid=True,
        gridcolor='lightgray',
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='black'
    ),
    updatemenus=[
        dict(
            buttons=dropdown_buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.12,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="white",
            bordercolor="gray",
            borderwidth=1
        )
    ],
    annotations=[
        dict(
            text="Select Year:",
            showarrow=False,
            x=0.01,
            y=1.15,
            xref="paper",
            yref="paper",
            align="left",
            font=dict(size=12)
        )
    ],
    showlegend=False,
    height=600,
    width=1400,
    plot_bgcolor='white',
    margin=dict(l=80, r=80, t=120, b=120)
)
#Display the results
fig.show()

print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Total countries analyzed: {df_avg_import['Country'].nunique()}")
print(f"Year range: {df_avg_import['Year'].min()} - {df_avg_import['Year'].max()}")
print(f"\nAverage Net Import Ratio across all countries and years: {df_avg_import['Net Import Ratio'].mean():.4f}")
print(f"Median Net Import Ratio: {df_avg_import['Net Import Ratio'].median():.4f}")
print(f"Min Net Import Ratio: {df_avg_import['Net Import Ratio'].min():.4f}")
print(f"Max Net Import Ratio: {df_avg_import['Net Import Ratio'].max():.4f}")

print("\n" + "="*60)
print("TOP 5 NET IMPORTERS (Average across all years)")
print("="*60)
top_importers = df_avg_import.groupby('Country')['Net Import Ratio'].mean().sort_values(ascending=False).head(5)
for country, ratio in top_importers.items():
    print(f"{country}: {ratio:.4f}")

print("\n" + "="*60)
print("TOP 5 NET EXPORTERS (Average across all years)")
print("="*60)
top_exporters = df_avg_import.groupby('Country')['Net Import Ratio'].mean().sort_values(ascending=True).head(5)
for country, ratio in top_exporters.items():
    print(f"{country}: {ratio:.4f}")


SUMMARY STATISTICS
Total countries analyzed: 32
Year range: 2010 - 2024

Average Net Import Ratio across all countries and years: 0.0680
Median Net Import Ratio: 0.0169
Min Net Import Ratio: -0.5088
Max Net Import Ratio: 0.9899

TOP 5 NET IMPORTERS (Average across all years)
Luxembourg: 0.8441
Lithuania: 0.7140
Hungary: 0.2838
Latvia: 0.2045
Finland: 0.1822

TOP 5 NET EXPORTERS (Average across all years)
Czech Republic: -0.2221
Sweden: -0.1568
France: -0.1159
Norway: -0.1038
Slovenia: -0.0855


###**4. LEARNING POINT OF THIS PROJECT**

Through this Python bootcamp project, we applied practical data analytics skills to real dataset. We have observed that data cleaning is a critical step for ensuring reliable insights and efficient analysis. We identified and handled such missing or inconsistent values and parsed the timestamp column into a standardized format for accurate tracking. Logical filtering and grouping were applied to restructure the dataset.


Using tools such as Pandas, NumPy, Matplotlib, Ipywidgets, and Plotly, we explored key energy indicators including production, final consumption, imports, exports to assess growth and volatility. Overall, this project strengthened our ability to think critically and transform datasets into clean analytical structures. Further, creating meaningful data driven analysis and decision-making.

#**THANK YOU**